# 02 — Hidden State Extraction (Colab)

**LLM Hallucination Detection Project — Phase 2**

This notebook extracts hidden-state representations from a frozen LLM (Llama-3.1-8B)
for both TruthfulQA and HaluEval datasets.

### Requirements
- **Runtime**: GPU (T4 free tier works with 4-bit quantization; A100 recommended for fp16)
- **HuggingFace account** with access to `meta-llama/Llama-3.1-8B` (accept Meta's license)
- **HF Token** for authentication

### What this notebook does
1. Installs dependencies & clones project repo
2. Authenticates with HuggingFace Hub
3. Loads Llama-3.1-8B (4-bit quantized for T4, fp16 for A100)
4. Registers forward hooks on all 32 transformer layers
5. Runs batched extraction on TruthfulQA (1,580 samples) and HaluEval (20,000 samples)
6. Saves hidden states to `hidden_states.h5` (HDF5, gzip compressed)
7. Downloads the file for local Phase 3 (probe training)

⏱️ **Estimated time**: ~15 min (TruthfulQA) + ~2.5 hrs (HaluEval) on T4

---
## Step 1 — Environment Setup

Install dependencies and clone the project repository.

In [3]:
# ── 1a. Install dependencies ──────────────────────────────────────────
!pip install -q torch transformers accelerate bitsandbytes datasets \
    huggingface_hub h5py pandas numpy scikit-learn tqdm

print("\n✅ Dependencies installed")


✅ Dependencies installed


In [4]:
!cd /content/llm_truth_probe && git pull

Already up to date.


In [5]:
# ── 1b. Clone project repository ─────────────────────────────────────
import os

REPO_URL = "https://github.com/mananparmar05/llm_truth_probe.git"
PROJECT_DIR = "/content/llm_truth_probe"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    print("✅ Repository cloned")
else:
    !cd {PROJECT_DIR} && git pull
    print("✅ Repository updated")

# Add project to Python path
import sys
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

print(f"Working directory: {os.getcwd()}")
!ls -la

Already up to date.
✅ Repository updated
Working directory: /content/llm_truth_probe
total 292
drwxr-xr-x 12 root root   4096 Aug 12 11:09 .
drwxr-xr-x  1 root root   4096 Aug 12 11:09 ..
drwxr-xr-x  2 root root   4096 Aug 12 11:09 analysis
drwxr-xr-x  2 root root   4096 Aug 12 11:09 configs
drwxr-xr-x  5 root root   4096 Aug 12 11:20 data
drwxr-xr-x  3 root root   4096 Aug 12 11:20 extraction
drwxr-xr-x  8 root root   4096 Aug 12 12:17 .git
-rw-r--r--  1 root root    507 Aug 12 11:09 .gitignore
-rw-r--r--  1 root root   7681 Aug 12 11:09 implementation_plan.md
drwxr-xr-x  2 root root   4096 Aug 12 11:09 inference
-rw-r--r--  1 root root 207249 Aug 12 11:09 LLM_Hallucination_Detection_Project.pdf
drwxr-xr-x  2 root root   4096 Aug 12 11:09 notebooks
drwxr-xr-x  2 root root   4096 Aug 12 11:09 probing
-rw-r--r--  1 root root   7401 Aug 12 11:09 README.md
-rw-r--r--  1 root root   2047 Aug 12 11:09 requirements.txt
drwxr-xr-x  3 root root   4096 Aug 12 11:09 results
-rwxr-xr-x  1 root ro

In [6]:
# Replace step 1c directly in your Colab cell with this safe version:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
  gpu_name = torch.cuda.get_device_name(0)
  props = torch.cuda.get_device_properties(0)
  gpu_bytes = getattr(props, "total_memory", getattr(props, "total_mem", 0))
  gpu_mem = gpu_bytes / 1024**3
  print(f"GPU:             {gpu_name}")
  print(f"GPU Memory:      {gpu_mem:.1f} GB")

  if gpu_mem >= 38:
    LOAD_IN_4BIT = False
    BATCH_SIZE = 8
    print("\n🟢 A100 detected — using fp16")
  else:
    LOAD_IN_4BIT = True
    BATCH_SIZE = 4
    print("\n🟡 T4 detected — using 4-bit quantization")
else:
  LOAD_IN_4BIT = True
  BATCH_SIZE = 4
  print("\n⚠️ No GPU detected")


PyTorch version: 2.11.0+cu128
CUDA available:  True
GPU:             Tesla T4
GPU Memory:      14.6 GB

🟡 T4 detected — using 4-bit quantization


---
## Step 2 — HuggingFace Authentication

**You need a HuggingFace token** with access to `meta-llama/Llama-3.1-8B`.

1. Go to https://huggingface.co/meta-llama/Llama-3.1-8B and accept Meta's license
2. Go to https://huggingface.co/settings/tokens and create a token (read access)
3. Paste it below when prompted

In [7]:
from huggingface_hub import login

# This will prompt you for your HF token
login()

print("\n✅ HuggingFace authentication successful")


✅ HuggingFace authentication successful


---
## Step 3 — Load the Model

Load Llama-3.1-8B with frozen parameters. On T4 we use 4-bit quantization;
on A100 we use fp16.

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import gc

MODEL_NAME = "Qwen/Qwen2.5-7B"

print(f"Loading model: {MODEL_NAME}")
print(f"Mode: {'4-bit quantized' if LOAD_IN_4BIT else 'fp16'}")
print(f"Batch size: {BATCH_SIZE}")
print("This may take 2-5 minutes...\n")

# ── Load tokenizer ────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"✅ Tokenizer loaded (vocab size: {tokenizer.vocab_size})")

# ── Load model ────────────────────────────────────────────────────────
if LOAD_IN_4BIT:
    # 4-bit quantization config for T4
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    # fp16 for A100
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )

model.eval()

# Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

print(f"\n✅ Model loaded successfully!")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")
print(f"   Device: {next(model.parameters()).device}")
print(f"   GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"   GPU memory total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Loading model: Qwen/Qwen2.5-7B
Mode: 4-bit quantized
Batch size: 4
This may take 2-5 minutes...

✅ Tokenizer loaded (vocab size: 151643)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


✅ Model loaded successfully!
   Parameters: 4.4B
   Device: cuda:0
   GPU memory used: 5.2 GB
   GPU memory total: 14.6 GB


---
## Step 4 — Load Preprocessed Datasets

Load the CSVs created in Phase 1. If they're not in the repo yet,
we'll download and create them from GitHub.

In [9]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

# Check if Phase 1 data exists
tqa_path = DATA_DIR / "truthfulqa" / "train.csv"
halu_path = DATA_DIR / "halueval" / "train.csv"

if not tqa_path.exists() or not halu_path.exists():
    print("⚠️  Phase 1 CSVs not found — running data preparation...")
    !python data/prepare_datasets.py
    print()

# Load all splits and combine into full datasets
def load_full_dataset(dataset_name):
    """Load train+val+test splits and return as list of (text, label) tuples."""
    dfs = []
    for split in ['train', 'val', 'test']:
        path = DATA_DIR / dataset_name / f'{split}.csv'
        dfs.append(pd.read_csv(path))
    full_df = pd.concat(dfs, ignore_index=True)
    samples = list(zip(full_df['text'].tolist(), full_df['label'].tolist()))
    return samples, full_df

tqa_samples, tqa_df = load_full_dataset('truthfulqa')
halu_samples, halu_df = load_full_dataset('halueval')

print(f"TruthfulQA: {len(tqa_samples):,} samples (label 0: {(tqa_df.label==0).sum()}, label 1: {(tqa_df.label==1).sum()})")
print(f"HaluEval:   {len(halu_samples):,} samples (label 0: {(halu_df.label==0).sum()}, label 1: {(halu_df.label==1).sum()})")
print(f"\nSample text: {tqa_samples[0][0][:100]}...")

TruthfulQA: 1,580 samples (label 0: 790, label 1: 790)
HaluEval:   20,000 samples (label 0: 10000, label 1: 10000)

Sample text: Question: What's on your mind right now?
Answer: I have no comment...


---
## Step 5 — Set Up Extraction Pipeline

Initialize the `HiddenStateExtractor`, `BatchRunner`, and `HiddenStateStore`
from the project's `extraction/` module.

In [10]:
from extraction.hook_extractor import HiddenStateExtractor
from extraction.batch_runner import BatchRunner
from extraction.storage import HiddenStateStore

import logging
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

# ── Initialize components ─────────────────────────────────────────────
extractor = HiddenStateExtractor(model, token_position="last")
print(f"\n✅ Extractor ready — {extractor.num_layers} layers detected, "
      f"hooking {extractor.num_hooked_layers} layers")

store = HiddenStateStore("data/hidden_states", format="hdf5")
print(f"✅ Store ready — will save to {store.path}")

runner = BatchRunner(
    model=model,
    tokenizer=tokenizer,
    extractor=extractor,
    batch_size=BATCH_SIZE,
    max_length=512,
)
print(f"✅ BatchRunner ready — batch_size={BATCH_SIZE}, max_length=512")

# Show estimated sizes
n_layers = extractor.num_layers
hidden_dim = 4096  # Llama-3.1-8B
bytes_per_sample = n_layers * hidden_dim * 4  # float32
tqa_size_mb = len(tqa_samples) * bytes_per_sample / 1024**2
halu_size_mb = len(halu_samples) * bytes_per_sample / 1024**2

print(f"\n📐 Estimated output sizes (before compression):")
print(f"   TruthfulQA: [{len(tqa_samples)}, {n_layers}, {hidden_dim}] → {tqa_size_mb:.0f} MB")
print(f"   HaluEval:   [{len(halu_samples)}, {n_layers}, {hidden_dim}] → {halu_size_mb:.0f} MB")
print(f"   Total:      {tqa_size_mb + halu_size_mb:.0f} MB (will be smaller with gzip)")


✅ Extractor ready — 28 layers detected, hooking 28 layers
✅ Store ready — will save to data/hidden_states/hidden_states.h5
✅ BatchRunner ready — batch_size=4, max_length=512

📐 Estimated output sizes (before compression):
   TruthfulQA: [1580, 28, 4096] → 691 MB
   HaluEval:   [20000, 28, 4096] → 8750 MB
   Total:      9441 MB (will be smaller with gzip)


---
## Step 6 — Extract Hidden States: TruthfulQA

Run extraction on the smaller dataset first (~15 min on T4).

In [ ]:
import time
import numpy as np
import gc
import torch

print("=" * 60)
print("  Extracting: TruthfulQA")
print(f"  Samples: {len(tqa_samples):,}")
print(
    f"  Expected shape: [{len(tqa_samples)}, {extractor.num_layers},"
    f" {hidden_dim}]"
)
print("=" * 60)

start = time.time()

tqa_hidden_states, tqa_labels = runner.run(
    samples=tqa_samples,
    store=store,
    dataset_name="truthfulqa",
)

elapsed = time.time() - start
print("\n✅ TruthfulQA extraction complete!")
print(f"   Shape: {tqa_hidden_states.shape}")
print(f"   Labels: {dict(zip(*np.unique(tqa_labels, return_counts=True)))}")
print(f"   Time: {elapsed/60:.1f} minutes")
print(f"   GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

# Free memory
import numpy as np

del tqa_hidden_states
gc.collect()
torch.cuda.empty_cache()


  Extracting: TruthfulQA
  Samples: 1,580
  Expected shape: [1580, 28, 4096]


Extracting [truthfulqa]:   5%|▍         | 18/395 [00:06<02:04,  3.04it/s]

---
## Step 7 — Extract Hidden States: HaluEval

Larger dataset — ~2.5 hours on T4. Progress bar shows batch-level progress.

> **⚡ Tip**: If you're on Colab free tier and worried about timeout,
> you can extract a subset first by changing `halu_samples[:5000]` below.

In [ ]:
import time
import numpy as np
import gc
import torch
# ── Option: extract full dataset or a subset ─────────────────────────
# For full extraction (recommended):
samples_to_extract = halu_samples

# For a quick test with subset (uncomment if needed):
# samples_to_extract = halu_samples[:5000]

print("="*60)
print("  Extracting: HaluEval")
print(f"  Samples: {len(samples_to_extract):,}")
print(f"  Expected shape: [{len(samples_to_extract)}, {extractor.num_layers}, 4096]")
print("="*60)

start = time.time()

halu_hidden_states, halu_labels = runner.run(
    samples=samples_to_extract,
    store=store,
    dataset_name="halueval",
)

elapsed = time.time() - start
print(f"\n✅ HaluEval extraction complete!")
print(f"   Shape: {halu_hidden_states.shape}")
print(f"   Labels: {dict(zip(*np.unique(halu_labels, return_counts=True)))}")
print(f"   Time: {elapsed/60:.1f} minutes")

# Free GPU memory
del halu_hidden_states
gc.collect()
torch.cuda.empty_cache()

---
## Step 8 — Verify & Inspect the HDF5 Store

In [1]:
import h5py
import numpy as np
import os

h5_path = "data/hidden_states/hidden_states.h5"

with h5py.File(h5_path, "r") as f:
    print("HDF5 Store Contents:")
    print("=" * 60)
    for ds_name in f.keys():
        grp = f[ds_name]
        hs_shape = grp["hidden_states"].shape
        labels_shape = grp["labels"].shape
        n_samples = grp.attrs.get("n_samples", "?")
        created = grp.attrs.get("created_at", "?")

        # Calculate compressed size
        hs_size_mb = grp["hidden_states"].id.get_storage_size() / 1024**2
        raw_size_mb = np.prod(hs_shape) * 4 / 1024**2  # float32

        print(f"\n  📦 {ds_name}")
        print(f"     hidden_states: {hs_shape}")
        print(f"     labels:        {labels_shape}")
        print(f"     created:       {created}")
        print(f"     compressed:    {hs_size_mb:.0f} MB (raw: {raw_size_mb:.0f} MB)")
        print(f"     compression:   {hs_size_mb/raw_size_mb:.1%} of original")

# File size on disk
file_size_mb = os.path.getsize(h5_path) / 1024**2
print(f"\n  📁 Total file size: {file_size_mb:.0f} MB")
print(f"\n✅ HDF5 store verified!")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'data/hidden_states/hidden_states.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
# ── Quick sanity check: load and inspect a slice ──────────────────────
with h5py.File(h5_path, "r") as f:
    # Load a small slice from TruthfulQA
    tqa_hs_sample = f["truthfulqa"]["hidden_states"][:5]   # first 5 samples
    tqa_labels_sample = f["truthfulqa"]["labels"][:5]

print("Sanity check (TruthfulQA, first 5 samples):")
print(f"  Shape: {tqa_hs_sample.shape}")
print(f"  Labels: {tqa_labels_sample}")
print(f"  Value range: [{tqa_hs_sample.min():.4f}, {tqa_hs_sample.max():.4f}]")
print(f"  Mean: {tqa_hs_sample.mean():.6f}")
print(f"  Std:  {tqa_hs_sample.std():.4f}")

# Check for NaN/Inf
print(f"  NaN count: {np.isnan(tqa_hs_sample).sum()}")
print(f"  Inf count: {np.isinf(tqa_hs_sample).sum()}")

assert not np.isnan(tqa_hs_sample).any(), "Found NaN values!"
assert not np.isinf(tqa_hs_sample).any(), "Found Inf values!"
print("\n✅ No NaN/Inf — hidden states look clean!")

---
## Step 9 — Download the HDF5 File

Download the hidden states file to your local machine for Phase 3 (probe training).

**On your local machine**, place the file at:
```
llm_truth_probe/data/hidden_states/hidden_states.h5
```

In [ ]:
# ── Option A: Download via Colab (works in browser) ──────────────────
try:
    from google.colab import files
    print("Downloading hidden_states.h5 ...")
    print(f"File size: {os.path.getsize(h5_path)/1024**2:.0f} MB")
    print("(This may take a few minutes for large files)\n")
    files.download(h5_path)
    print("\n✅ Download started! Check your browser's download folder.")
except ImportError:
    print("Not running in Colab — use the manual download instructions below.")

In [ ]:
# ── Option B: Save to Google Drive (recommended for large files) ─────
# Uncomment the lines below to mount Google Drive and copy the file.

# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# drive_dest = '/content/drive/MyDrive/llm_truth_probe/hidden_states.h5'
# os.makedirs(os.path.dirname(drive_dest), exist_ok=True)
# shutil.copy2(h5_path, drive_dest)
# print(f"✅ Copied to Google Drive: {drive_dest}")
# print(f"   File size: {os.path.getsize(drive_dest)/1024**2:.0f} MB")

---
## ✅ Phase 2 Complete!

### What was extracted
| Dataset | Samples | Shape | Description |
|---------|---------|-------|-------------|
| TruthfulQA | 1,580 | [1580, 32, 4096] | 790 truthful + 790 hallucinated |
| HaluEval | 20,000 | [20000, 32, 4096] | 10,000 truthful + 10,000 hallucinated |

### Next steps (Phase 3 — runs locally, no GPU needed!)

1. Download `hidden_states.h5` to your local project at `data/hidden_states/`
2. Run `notebooks/03_probe_training.ipynb` locally
3. This trains Logistic Regression, MLP, and SVM probes on each of the 32 layers
4. Expected best layer: ~Layer 14-16 with AUROC ~0.94-0.96

### Optional: Secondary Model (Mistral-7B)

To extract hidden states from Mistral-7B for cross-model comparison,
re-run this notebook changing `MODEL_NAME` to `mistralai/Mistral-7B-v0.1`
and the store dataset names to `truthfulqa_mistral` / `halueval_mistral`.